In [23]:
from openpyxl import load_workbook
from collections import defaultdict
import pandas as pd

In [24]:
archivo = "../../pilot_automation_v1/master_data/ZonificacIón_Primera_Infancia_Hasta_Oct.xlsx"

# HOJA ORIGEN
hoja_origen = "Comunitarios"

# HOJA NUEVA
hoja_salida = "Comunitarios_agrupado"

In [26]:
# Leer hoja
df = pd.read_excel(archivo, sheet_name=hoja_origen)

# Columnas por las que se agrupa
columnas_grupo = ["Regional UDS", "Centro Zonal UDS", "ZONA", "Municipio UDS", "Nombre Servicio"]
str_columns = ["Código UDS"]

# Definir cómo agregar cada columna
agg_dict = {}

for col in df.columns:
    if col in columnas_grupo:
        continue

    # Columnas numéricas: sumar
    if pd.api.types.is_numeric_dtype(df[col]) and col not in str_columns:
        agg_dict[col] = "sum"

    # Columnas no numéricas: dejar valores únicos
    else:
        agg_dict[col] = lambda x: " | ".join(
            sorted(set(str(v) for v in x.dropna() if str(v).strip() != ""))
        )

# Agrupar
df_agrupado = (
    df.groupby(columnas_grupo, dropna=False)
      .agg(agg_dict)
      .reset_index()
)

# Ordenar para mantener las columnas de la hoja original
columnas_finales = [col for col in df.columns if col]
df_agrupado = df_agrupado[columnas_finales]

# Guardar en el mismo Excel, creando/reemplazando la hoja agrupada
with pd.ExcelWriter(
    archivo,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df_agrupado.to_excel(writer, sheet_name=hoja_salida, index=False)

print("Proceso terminado.")
print(f"Hoja creada o reemplazada: {hoja_salida}")

Proceso terminado.
Hoja creada o reemplazada: Comunitarios_agrupado
